# Motion Tracking for Reinforcement Learning in MuJoCo

This tutorial covers the essential MuJoCo knowledge needed for **motion tracking RL** tasks, as used in works like [DeepMimic](https://xbpeng.github.io/projects/DeepMimic/index.html), [BeyondMimic](https://beyondmimic.github.io/), and various humanoid locomotion papers.

Motion tracking RL trains a policy $\pi(a|s)$ to control a simulated character so that it **imitates a reference motion** (e.g., from motion capture data). The core loop is:

1. At each timestep, the agent observes the **current state** $s_t$ and a **reference target** $s_t^{\text{ref}}$
2. The agent outputs **actions** $a_t$ (typically target joint angles or torques)
3. The simulator advances one step via physics
4. A **tracking reward** $r_t$ measures how closely the simulated pose matches the reference

This notebook covers the MuJoCo-specific knowledge required at each step:

| Section | Topic |
|---------|-------|
| 1 | Humanoid model anatomy: bodies, joints, actuators |
| 2 | State representation: `qpos`, `qvel`, and the free joint |
| 3 | Forward kinematics: accessing body positions & orientations |
| 4 | Quaternion math: MuJoCo's orientation utilities |
| 5 | Actuator models: torque vs. position control, PD controllers |
| 6 | Reference motion: representing and visualizing trajectories |
| 7 | Reward design: DeepMimic-style tracking rewards |
| 8 | Complete motion tracking demo |
| 9 | RL integration: observation / action space design |

In [ ]:
import mujoco
import mediapy as media
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## 1. Humanoid Model Anatomy

A humanoid model in MuJoCo is defined in MJCF (XML). The key components are:

- **Bodies**: rigid links forming the kinematic tree (torso, thigh, shin, foot, ...)
- **Joints**: degrees of freedom connecting bodies. Types include:
  - `free`: 6-DoF root joint (3 translation + 3 rotation), stored as 7 values in qpos (3 pos + 4 quat)
  - `hinge`: 1-DoF revolute joint (1 angle in qpos)
  - `ball`: 3-DoF spherical joint (4 quat values in qpos)
- **Actuators**: motors that apply forces/torques to joints
- **Geoms**: collision/visual geometry attached to bodies

Below we define a standard humanoid with:
- A **free joint** at the root (torso)
- **Hinge joints** for head, abdomen (3-DoF), hips (3-DoF each), knees, ankles (2-DoF each), shoulders (2-DoF each), elbows
- **Motor actuators** on each hinge joint

In [ ]:
HUMANOID_XML = """
<mujoco model="humanoid">
  <compiler angle="degree" inertiafromgeom="true"/>
  <option integrator="RK4" timestep="0.002"/>

  <default>
    <joint limited="true" damping="1" armature="0"/>
    <geom condim="3" material="body_mat"/>
    <motor ctrllimited="true" ctrlrange="-1 1"/>
  </default>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1=".4 .5 .6" rgb2="0 0 0" width="100" height="100"/>
    <texture name="texgeom" type="cube" builtin="flat" mark="cross" width="127" height="1278"
             rgb1="0.8 0.6 0.4" rgb2="0.8 0.6 0.4" markrgb="1 1 1" random="0.01"/>
    <texture name="texplane" type="2d" builtin="checker" rgb1=".2 .3 .4" rgb2=".1 .15 .2"
             width="512" height="512"/>
    <material name="body_mat" texture="texgeom" texuniform="true"/>
    <material name="MatPlane" texture="texplane" texrepeat="1 1" reflectance="0.5"/>
  </asset>

  <worldbody>
    <light diffuse=".5 .5 .5" pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="10 10 0.1" material="MatPlane" condim="3"/>

    <body name="torso" pos="0 0 1.4">
      <freejoint name="root"/>
      <geom name="torso_geom" type="capsule" fromto="0 -.07 0 0 .07 0" size="0.07" density="1000"/>
      <geom name="uwaist" type="capsule" fromto="-.01 -.06 -.12 -.01 .06 -.12" size="0.06" density="1000"/>

      <body name="head" pos="0 0 .19">
        <joint name="head" type="hinge" pos="0 0 0" axis="0 1 0" range="-45 45" damping="5"/>
        <geom name="head_geom" type="sphere" size="0.09" density="900"/>
      </body>

      <body name="lwaist" pos="-.01 0 -0.260">
        <joint name="abdomen_z" type="hinge" pos="0 0 0.065" axis="0 0 1" range="-45 45" damping="5" stiffness="20" armature="0.02"/>
        <joint name="abdomen_y" type="hinge" pos="0 0 0.065" axis="0 1 0" range="-75 30" damping="5" stiffness="10" armature="0.02"/>
        <geom name="lwaist_geom" type="capsule" fromto="0 -.06 0 0 .06 0" size="0.06" density="1000"/>
        <body name="pelvis" pos="0 0 -0.165">
          <joint name="abdomen_x" type="hinge" pos="0 0 0.1" axis="1 0 0" range="-35 35" damping="5" stiffness="10" armature="0.02"/>
          <geom name="butt" type="capsule" fromto="-.02 -.07 0 -.02 .07 0" size="0.09" density="1000"/>

          <!-- Right leg -->
          <body name="right_thigh" pos="0 -0.1 -0.04">
            <joint name="right_hip_x" type="hinge" pos="0 0 0" axis="1 0 0" range="-25 5" damping="5" stiffness="10" armature="0.01"/>
            <joint name="right_hip_z" type="hinge" pos="0 0 0" axis="0 0 1" range="-60 35" damping="5" stiffness="10" armature="0.01"/>
            <joint name="right_hip_y" type="hinge" pos="0 0 0" axis="0 1 0" range="-110 20" damping="5" stiffness="20" armature="0.01"/>
            <geom name="right_thigh_geom" type="capsule" fromto="0 0 0 0 0.01 -0.34" size="0.06" density="1000"/>
            <body name="right_shin" pos="0 0.01 -0.403">
              <joint name="right_knee" type="hinge" pos="0 0 .02" axis="0 -1 0" range="-160 -2" stiffness="5" armature="0.006"/>
              <geom name="right_shin_geom" type="capsule" fromto="0 0 0 0 0 -.3" size="0.049" density="1000"/>
              <body name="right_foot" pos="0 0 -0.35">
                <joint name="right_ankle_y" type="hinge" pos="0 0 0.08" axis="0 1 0" range="-50 50" stiffness="4" armature="0.0008"/>
                <joint name="right_ankle_x" type="hinge" pos="0 0 0.08" axis="1 0 0.5" range="-50 50" stiffness="1" armature="0.0006"/>
                <geom name="right_foot_geom" type="capsule" fromto="-.07 -0.02 0 0.14 -0.04 0" size="0.027" density="1100"/>
              </body>
            </body>
          </body>

          <!-- Left leg -->
          <body name="left_thigh" pos="0 0.1 -0.04">
            <joint name="left_hip_x" type="hinge" pos="0 0 0" axis="1 0 0" range="-25 5" damping="5" stiffness="10" armature="0.01"/>
            <joint name="left_hip_z" type="hinge" pos="0 0 0" axis="0 0 1" range="-60 35" damping="5" stiffness="10" armature="0.01"/>
            <joint name="left_hip_y" type="hinge" pos="0 0 0" axis="0 1 0" range="-110 20" damping="5" stiffness="20" armature="0.01"/>
            <geom name="left_thigh_geom" type="capsule" fromto="0 0 0 0 -0.01 -0.34" size="0.06" density="1000"/>
            <body name="left_shin" pos="0 -0.01 -0.403">
              <joint name="left_knee" type="hinge" pos="0 0 .02" axis="0 -1 0" range="-160 -2" stiffness="5" armature="0.006"/>
              <geom name="left_shin_geom" type="capsule" fromto="0 0 0 0 0 -.3" size="0.049" density="1000"/>
              <body name="left_foot" pos="0 0 -0.35">
                <joint name="left_ankle_y" type="hinge" pos="0 0 0.08" axis="0 1 0" range="-50 50" stiffness="4" armature="0.0008"/>
                <joint name="left_ankle_x" type="hinge" pos="0 0 0.08" axis="1 0 0.5" range="-50 50" stiffness="1" armature="0.0006"/>
                <geom name="left_foot_geom" type="capsule" fromto="-.07 0.02 0 0.14 0.04 0" size="0.027" density="1100"/>
              </body>
            </body>
          </body>
        </body>
      </body>

      <!-- Right arm -->
      <body name="right_upper_arm" pos="0 -0.17 0.06">
        <joint name="right_shoulder1" type="hinge" pos="0 0 0" axis="2 1 1" range="-85 60" damping="5" stiffness="1" armature="0.0068"/>
        <joint name="right_shoulder2" type="hinge" pos="0 0 0" axis="0 -1 1" range="-85 60" damping="5" stiffness="1" armature="0.0051"/>
        <geom name="right_uarm_geom" type="capsule" fromto="0 0 0 .16 -.16 -.16" size="0.04" density="1000"/>
        <body name="right_lower_arm" pos=".18 -.18 -.18">
          <joint name="right_elbow" type="hinge" pos="0 0 0" axis="0 -1 1" range="-90 50" stiffness="0" armature="0.0028"/>
          <geom name="right_larm_geom" type="capsule" fromto="0.01 0.01 0.01 .17 .17 .17" size="0.031" density="1000"/>
          <site name="right_hand" pos=".17 .17 .17" size="0.02" rgba="1 0 0 1"/>
        </body>
      </body>

      <!-- Left arm -->
      <body name="left_upper_arm" pos="0 0.17 0.06">
        <joint name="left_shoulder1" type="hinge" pos="0 0 0" axis="2 -1 1" range="-60 85" damping="5" stiffness="1" armature="0.0068"/>
        <joint name="left_shoulder2" type="hinge" pos="0 0 0" axis="0 1 1" range="-60 85" damping="5" stiffness="1" armature="0.0051"/>
        <geom name="left_uarm_geom" type="capsule" fromto="0 0 0 .16 .16 -.16" size="0.04" density="1000"/>
        <body name="left_lower_arm" pos=".18 .18 -.18">
          <joint name="left_elbow" type="hinge" pos="0 0 0" axis="0 -1 -1" range="-90 50" stiffness="0" armature="0.0028"/>
          <geom name="left_larm_geom" type="capsule" fromto="0.01 -0.01 0.01 .17 -.17 .17" size="0.031" density="1000"/>
          <site name="left_hand" pos=".17 -.17 .17" size="0.02" rgba="1 0 0 1"/>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="abdomen_z_act" joint="abdomen_z" gear="100"/>
    <motor name="abdomen_y_act" joint="abdomen_y" gear="100"/>
    <motor name="abdomen_x_act" joint="abdomen_x" gear="100"/>
    <motor name="right_hip_x_act" joint="right_hip_x" gear="100"/>
    <motor name="right_hip_z_act" joint="right_hip_z" gear="100"/>
    <motor name="right_hip_y_act" joint="right_hip_y" gear="200"/>
    <motor name="right_knee_act" joint="right_knee" gear="150"/>
    <motor name="right_ankle_y_act" joint="right_ankle_y" gear="50"/>
    <motor name="right_ankle_x_act" joint="right_ankle_x" gear="50"/>
    <motor name="left_hip_x_act" joint="left_hip_x" gear="100"/>
    <motor name="left_hip_z_act" joint="left_hip_z" gear="100"/>
    <motor name="left_hip_y_act" joint="left_hip_y" gear="200"/>
    <motor name="left_knee_act" joint="left_knee" gear="150"/>
    <motor name="left_ankle_y_act" joint="left_ankle_y" gear="50"/>
    <motor name="left_ankle_x_act" joint="left_ankle_x" gear="50"/>
    <motor name="right_shoulder1_act" joint="right_shoulder1" gear="25"/>
    <motor name="right_shoulder2_act" joint="right_shoulder2" gear="25"/>
    <motor name="right_elbow_act" joint="right_elbow" gear="25"/>
    <motor name="left_shoulder1_act" joint="left_shoulder1" gear="25"/>
    <motor name="left_shoulder2_act" joint="left_shoulder2" gear="25"/>
    <motor name="left_elbow_act" joint="left_elbow" gear="25"/>
    <motor name="head_act" joint="head" gear="10"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(HUMANOID_XML)
data = mujoco.MjData(model)

print(f"Degrees of freedom (nv): {model.nv}")
print(f"Generalized coordinates (nq): {model.nq}")
print(f"Actuators (nu): {model.nu}")
print(f"Bodies (nbody): {model.nbody}")
print(f"Joints (njnt): {model.njnt}")
print()
print("--- Joint Layout ---")
type_names = {0: 'free', 1: 'ball', 2: 'slide', 3: 'hinge'}
qpos_sizes = {0: 7, 1: 4, 2: 1, 3: 1}
qpos_offset = 0
for i in range(model.njnt):
    jtype = model.jnt_type[i]
    nq = qpos_sizes[jtype]
    print(f"  joint {i:2d}: {model.joint(i).name:20s} type={type_names[jtype]:5s}  qpos[{qpos_offset}:{qpos_offset+nq}]")
    qpos_offset += nq

In [ ]:
# Visualize the humanoid in its initial pose
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data)
media.show_image(renderer.render())

## 2. State Representation: `qpos`, `qvel` and the Free Joint

In motion tracking RL, the **state** of the character is fully described by two vectors:

| Vector | Description | Dimension |
|--------|-------------|----------|
| `data.qpos` | Generalized positions | `model.nq` |
| `data.qvel` | Generalized velocities | `model.nv` |

**Key insight**: `nq != nv` because the free joint uses a **quaternion** (4 values) for orientation in `qpos`, but only **3 angular velocity** values in `qvel`.

For our humanoid:
- `qpos` has 29 dimensions: 7 (free joint: 3 pos + 4 quat) + 22 (hinge joints: 1 each)
- `qvel` has 28 dimensions: 6 (free joint: 3 lin vel + 3 ang vel) + 22 (hinge joints: 1 each)

### Free joint qpos layout
```
qpos[0:3]  = (x, y, z)     — root position in world frame
qpos[3:7]  = (w, x, y, z)  — root orientation as unit quaternion (MuJoCo convention: w first)
qpos[7:]   = joint angles   — one per hinge joint (radians)
```

### Free joint qvel layout
```
qvel[0:3]  = (vx, vy, vz)      — root linear velocity in world frame
qvel[3:6]  = (wx, wy, wz)      — root angular velocity in world frame
qvel[6:]   = joint velocities   — one per hinge joint (rad/s)
```

In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

print(f"qpos shape: {data.qpos.shape}  (nq={model.nq})")
print(f"qvel shape: {data.qvel.shape}  (nv={model.nv})")
print()

# Decompose qpos
root_pos = data.qpos[:3]
root_quat = data.qpos[3:7]  # (w, x, y, z)
joint_angles = data.qpos[7:]

print(f"Root position:    {root_pos}")
print(f"Root quaternion:  {root_quat}  (identity = [1,0,0,0])")
print(f"Joint angles:     {joint_angles}  (all zeros at reset)")
print()

# Decompose qvel
root_lin_vel = data.qvel[:3]
root_ang_vel = data.qvel[3:6]
joint_vels = data.qvel[6:]

print(f"Root linear vel:  {root_lin_vel}")
print(f"Root angular vel: {root_ang_vel}")
print(f"Joint velocities: {joint_vels}")

In [ ]:
# Demonstrate: directly setting qpos to pose the character
# This is essential for motion tracking — we need to set the model to a reference pose.

mujoco.mj_resetData(model, data)

# Bend the right knee (joint index 8 -> qpos index 7+8-1=14, since joint 0 is free)
# Let's find the qpos address properly using the model
right_knee_jnt_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'right_knee')
right_knee_qpos_adr = model.jnt_qposadr[right_knee_jnt_id]
print(f"right_knee joint id: {right_knee_jnt_id}, qpos address: {right_knee_qpos_adr}")

# Set a bent knee angle (in radians, since MuJoCo uses radians internally)
data.qpos[right_knee_qpos_adr] = np.deg2rad(-90)

# Raise the right arm
rs1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'right_shoulder1')
data.qpos[model.jnt_qposadr[rs1_id]] = np.deg2rad(40)

mujoco.mj_forward(model, data)
renderer.update_scene(data)
media.show_image(renderer.render())

## 3. Forward Kinematics: Body Positions & Orientations

After calling `mj_forward(model, data)` or `mj_step(model, data)`, MuJoCo computes all **derived quantities** from the current state. For motion tracking, the most important are:

| Data field | Description | Shape per body |
|-----------|-------------|----------------|
| `data.xpos[body_id]` | Body position in world frame | (3,) |
| `data.xquat[body_id]` | Body orientation quaternion | (4,) — (w,x,y,z) |
| `data.xmat[body_id]` | Body orientation as 3×3 rotation matrix (flattened) | (9,) |
| `data.cvel[body_id]` | Body 6D velocity (angular, linear) in world frame | (6,) |
| `data.subtree_com[body_id]` | Center of mass of the subtree rooted at body | (3,) |
| `data.site_xpos[site_id]` | Site position in world frame | (3,) |

These are used in reward computation (e.g., comparing body positions/orientations between simulated and reference).

In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

print("--- Body positions (xpos) ---")
for i in range(model.nbody):
    name = model.body(i).name
    pos = data.xpos[i]
    quat = data.xquat[i]
    print(f"  {name:20s}  pos={pos}  quat={quat}")

print()
print("--- End-effector sites ---")
for name in ['right_hand', 'left_hand']:
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, name)
    print(f"  {name}: pos={data.site_xpos[sid]}")

print(f"\n--- Center of mass ---")
print(f"  Whole-body CoM: {data.subtree_com[0]}")

## 4. Quaternion Math in MuJoCo

Orientation tracking is at the heart of motion imitation. MuJoCo uses **quaternions** in `(w, x, y, z)` convention and provides utility functions:

| Function | Description |
|----------|-------------|
| `mju_mulQuat(res, q1, q2)` | Quaternion multiplication: `res = q1 * q2` |
| `mju_negQuat(res, q)` | Quaternion conjugate/inverse: `res = q*` |
| `mju_quat2Vel(vel, q, dt)` | Convert quaternion difference to angular velocity |
| `mju_mat2Quat(quat, mat)` | Convert 3×3 rotation matrix to quaternion |
| `mju_quat2Mat(mat, quat)` | Convert quaternion to 3×3 rotation matrix |
| `mju_axisAngle2Quat(quat, axis, angle)` | Axis-angle to quaternion |
| `mju_subQuat(res, qa, qb)` | Compute rotation from `qb` to `qa`: equivalent to angular difference |

### Computing orientation error

To measure how different two orientations are (e.g., simulated vs. reference body orientation), we compute:

$$q_{\text{error}} = q_{\text{sim}} \otimes q_{\text{ref}}^{-1}$$

Then convert to an angular velocity vector via `mju_quat2Vel`. The norm of this vector gives the orientation error in radians.

In [ ]:
def quat_error(q_sim, q_ref):
    """Compute orientation error between two quaternions as an angular velocity vector.
    Returns a 3D vector whose norm is the angle of rotation (in radians) between the two orientations.
    """
    q_ref_conj = np.zeros(4)
    q_err = np.zeros(4)
    vel = np.zeros(3)
    
    mujoco.mju_negQuat(q_ref_conj, q_ref)       # q_ref_conj = q_ref^{-1}
    mujoco.mju_mulQuat(q_err, q_sim, q_ref_conj) # q_err = q_sim * q_ref^{-1}
    mujoco.mju_quat2Vel(vel, q_err, 1.0)         # convert to angular velocity (dt=1 -> gives angle directly)
    return vel

# Example: identity vs 90-degree rotation around Z
q_identity = np.array([1.0, 0.0, 0.0, 0.0])
q_90z = np.zeros(4)
mujoco.mju_axisAngle2Quat(q_90z, np.array([0.0, 0.0, 1.0]), np.pi/2)

err = quat_error(q_90z, q_identity)
print(f"q_identity:   {q_identity}")
print(f"q_90z:        {q_90z}")
print(f"Error vector: {err}")
print(f"Error angle:  {np.linalg.norm(err):.4f} rad = {np.degrees(np.linalg.norm(err)):.1f} deg")

# Verify: same quaternion -> zero error
err_zero = quat_error(q_identity, q_identity)
print(f"\nSame quat error: {err_zero}  (should be zeros)")

# mju_subQuat: an alternative that directly gives the angular difference
sub_res = np.zeros(3)
mujoco.mju_subQuat(sub_res, q_90z, q_identity)
print(f"mju_subQuat result: {sub_res}  (should match error vector above)")

## 5. Actuator Models: Torque Control vs. Position Control

In motion tracking RL, the agent's **action** is typically either:

1. **Torque control** (`motor` actuator): action = normalized torque, applied as `gear * ctrl`. Simple but requires the policy to learn stabilization.
2. **Position control** (`position` actuator): action = target joint angle. A built-in PD controller tracks this target. More stable but limits what the policy can express.
3. **PD target** (common in DeepMimic/BeyondMimic): the policy outputs **target joint angles** $q^{\text{target}}$, and a PD controller computes torques:
   $$\tau = k_p (q^{\text{target}} - q) - k_d \dot{q}$$

Our model uses **motor actuators** (torque control). Let's also show how to implement a PD controller on top.

In [ ]:
print("--- Actuator info ---")
for i in range(model.nu):
    name = model.actuator(i).name
    gear = model.actuator_gear[i, 0]
    ctrlrange = model.actuator_ctrlrange[i]
    print(f"  {name:25s}  gear={gear:6.0f}  ctrlrange=[{ctrlrange[0]:.1f}, {ctrlrange[1]:.1f}]")

In [ ]:
def pd_controller(model, data, target_qpos, kp, kd):
    """Compute torques using a PD controller.
    
    This is the standard approach in motion tracking RL:
    - The RL policy outputs target_qpos (desired joint angles)
    - The PD controller converts them to torques
    
    Args:
        target_qpos: desired joint angles, shape (n_actuated_joints,)
        kp: proportional gains, shape (n_actuated_joints,)
        kd: derivative gains, shape (n_actuated_joints,)
    Returns:
        torques: shape (n_actuated_joints,)
    """
    # Current joint angles and velocities (skip the free joint)
    current_qpos = data.qpos[7:]  # skip free joint (7 values)
    current_qvel = data.qvel[6:]  # skip free joint (6 values)
    
    # PD control law
    position_error = target_qpos - current_qpos
    torques = kp * position_error - kd * current_qvel
    
    return torques

# Example: PD gains (typical values for humanoid)
n_joints = model.nu
kp = np.full(n_joints, 100.0)
kd = np.full(n_joints, 10.0)

# Test: try to hold the initial pose
mujoco.mj_resetData(model, data)
target = data.qpos[7:].copy()  # target = initial pose

frames = []
for step in range(500):
    torques = pd_controller(model, data, target, kp, kd)
    # Normalize torques to [-1, 1] range using actuator gear ratios
    gears = model.actuator_gear[:, 0]
    data.ctrl[:] = np.clip(torques / gears, -1, 1)
    
    mujoco.mj_step(model, data)
    if step % 10 == 0:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

media.show_video(frames, fps=25, title='PD controller holding initial pose')

## 6. Reference Motion: Representing & Visualizing Trajectories

In motion tracking RL, a **reference motion** is a time-indexed sequence of target poses:

$$\{(q^{\text{ref}}_t, \dot{q}^{\text{ref}}_t)\}_{t=0}^{T}$$

In practice, this comes from:
- **Motion capture (MoCap)** data retargeted to the simulation model
- **Kinematic trajectory optimization**
- **Hand-crafted keyframes** (for simple motions)

Each frame contains a full `qpos` vector. For the free joint, this includes the root position and orientation.

Below we generate a **simple walking-like reference motion** using keyframe interpolation to demonstrate the concept.

In [ ]:
def get_joint_qpos_idx(model, joint_name):
    """Get the qpos index for a named joint."""
    jnt_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    return model.jnt_qposadr[jnt_id]

def create_reference_motion(model, duration=2.0, dt=0.01):
    """Generate a simple cyclic leg-swing reference motion.
    
    This simulates a simplified walking gait by oscillating hip and knee joints.
    In real applications, this would be loaded from MoCap data.
    
    Returns:
        times: array of shape (T,)
        ref_qpos: array of shape (T, nq) — full qpos at each timestep
    """
    T = int(duration / dt)
    times = np.arange(T) * dt
    ref_qpos = np.zeros((T, model.nq))
    
    # Base pose: standing
    base_qpos = np.zeros(model.nq)
    base_qpos[2] = 1.4    # root height
    base_qpos[3] = 1.0    # quaternion w (identity)
    
    freq = 2.0 * np.pi / duration  # one full cycle
    
    # Joint indices
    rhy = get_joint_qpos_idx(model, 'right_hip_y')
    rk  = get_joint_qpos_idx(model, 'right_knee')
    ray = get_joint_qpos_idx(model, 'right_ankle_y')
    lhy = get_joint_qpos_idx(model, 'left_hip_y')
    lk  = get_joint_qpos_idx(model, 'left_knee')
    lay = get_joint_qpos_idx(model, 'left_ankle_y')
    rs1 = get_joint_qpos_idx(model, 'right_shoulder1')
    ls1 = get_joint_qpos_idx(model, 'left_shoulder1')
    
    for i, t in enumerate(times):
        qpos = base_qpos.copy()
        phase = freq * t
        
        # Forward displacement
        qpos[0] = 0.3 * t  # slow forward motion
        
        # Hip swing: right and left legs in anti-phase
        hip_amp = np.deg2rad(30)
        qpos[rhy] = -hip_amp * np.sin(phase)
        qpos[lhy] = hip_amp * np.sin(phase)
        
        # Knee bend: bend during swing phase
        knee_amp = np.deg2rad(40)
        qpos[rk] = -knee_amp * np.maximum(0, np.sin(phase))  # bend when swinging forward
        qpos[lk] = -knee_amp * np.maximum(0, -np.sin(phase))
        
        # Ankle compensation
        ankle_amp = np.deg2rad(10)
        qpos[ray] = ankle_amp * np.sin(phase)
        qpos[lay] = -ankle_amp * np.sin(phase)
        
        # Arm swing (opposite to legs)
        arm_amp = np.deg2rad(20)
        qpos[rs1] = arm_amp * np.sin(phase)
        qpos[ls1] = -arm_amp * np.sin(phase)
        
        ref_qpos[i] = qpos
    
    return times, ref_qpos

ref_times, ref_qpos = create_reference_motion(model, duration=2.0, dt=0.01)
print(f"Reference motion: {len(ref_times)} frames, duration={ref_times[-1]:.2f}s")
print(f"qpos shape per frame: {ref_qpos[0].shape}")

In [ ]:
# Visualize the reference motion by directly setting qpos
frames_ref = []
for i in range(0, len(ref_qpos), 5):  # every 5th frame
    data.qpos[:] = ref_qpos[i]
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)
    frames_ref.append(renderer.render().copy())

media.show_video(frames_ref, fps=20, title='Reference motion (kinematic playback)')

In [ ]:
# Plot reference joint trajectories
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

joints_to_plot = [
    ('right_hip_y', 'Right Hip Y'),
    ('right_knee', 'Right Knee'),
    ('left_hip_y', 'Left Hip Y'),
    ('left_knee', 'Left Knee'),
]

for ax, (jname, label) in zip(axes.flat, joints_to_plot):
    idx = get_joint_qpos_idx(model, jname)
    ax.plot(ref_times, np.degrees(ref_qpos[:, idx]))
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (deg)')
    ax.set_title(label)
    ax.grid(True, alpha=0.3)

plt.suptitle('Reference Motion Joint Trajectories', fontsize=14)
plt.tight_layout()
plt.savefig('imgs/ref_motion_trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Reward Design: DeepMimic-style Tracking Rewards

The reward function is the core of motion tracking RL. Following the DeepMimic paper, the total reward is a weighted sum of tracking terms:

$$r_t = w_q \cdot r_q + w_v \cdot r_v + w_e \cdot r_e + w_c \cdot r_c$$

where:

| Term | Formula | Tracks |
|------|---------|--------|
| $r_q$ | $\exp\left(-k_q \sum_j \|\hat{q}_j \ominus q_j^{\text{ref}}\|^2\right)$ | Joint orientations |
| $r_v$ | $\exp\left(-k_v \sum_j \|\dot{q}_j - \dot{q}_j^{\text{ref}}\|^2\right)$ | Joint velocities |
| $r_e$ | $\exp\left(-k_e \sum_e \|p_e - p_e^{\text{ref}}\|^2\right)$ | End-effector positions |
| $r_c$ | $\exp\left(-k_c \|p_{\text{com}} - p_{\text{com}}^{\text{ref}}\|^2\right)$ | Center of mass |

The $\exp(-k \cdot \text{error}^2)$ form ensures rewards are in $[0, 1]$ and fall off smoothly.

In [ ]:
class MotionTrackingReward:
    """DeepMimic-style motion tracking reward computation.
    
    This class encapsulates the reward function used in motion imitation RL.
    """
    
    def __init__(self, model,
                 w_qpos=0.5, w_qvel=0.1, w_end_effector=0.15, w_com=0.1, w_root_orient=0.15,
                 k_qpos=2.0, k_qvel=0.1, k_end_effector=40.0, k_com=10.0, k_root_orient=5.0,
                 end_effector_sites=None):
        self.model = model
        
        # Reward weights (should sum to 1)
        self.w_qpos = w_qpos
        self.w_qvel = w_qvel
        self.w_ee = w_end_effector
        self.w_com = w_com
        self.w_root = w_root_orient
        
        # Kernel widths (control sensitivity)
        self.k_qpos = k_qpos
        self.k_qvel = k_qvel
        self.k_ee = k_end_effector
        self.k_com = k_com
        self.k_root = k_root_orient
        
        # End-effector site IDs
        if end_effector_sites is None:
            end_effector_sites = ['right_hand', 'left_hand']
        self.ee_site_ids = [
            mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, name)
            for name in end_effector_sites
        ]
    
    def compute(self, data, ref_qpos, ref_qvel, ref_data):
        """Compute the tracking reward.
        
        Args:
            data: current MjData (after mj_forward)
            ref_qpos: reference qpos
            ref_qvel: reference qvel
            ref_data: MjData with reference pose set (after mj_forward)
        Returns:
            total_reward, reward_dict
        """
        rewards = {}
        
        # 1. Joint angle reward (for hinge joints, skip free joint)
        joint_diff = data.qpos[7:] - ref_qpos[7:]
        rewards['qpos'] = np.exp(-self.k_qpos * np.sum(joint_diff ** 2))
        
        # 2. Joint velocity reward
        vel_diff = data.qvel[6:] - ref_qvel[6:]
        rewards['qvel'] = np.exp(-self.k_qvel * np.sum(vel_diff ** 2))
        
        # 3. End-effector position reward
        ee_error = 0.0
        for sid in self.ee_site_ids:
            diff = data.site_xpos[sid] - ref_data.site_xpos[sid]
            ee_error += np.sum(diff ** 2)
        rewards['end_effector'] = np.exp(-self.k_ee * ee_error)
        
        # 4. Center of mass reward
        com_diff = data.subtree_com[0] - ref_data.subtree_com[0]
        rewards['com'] = np.exp(-self.k_com * np.sum(com_diff ** 2))
        
        # 5. Root orientation reward
        root_ori_err = quat_error(data.xquat[1], ref_data.xquat[1])  # body 1 = torso
        rewards['root_orient'] = np.exp(-self.k_root * np.sum(root_ori_err ** 2))
        
        # Weighted sum
        total = (self.w_qpos * rewards['qpos'] +
                 self.w_qvel * rewards['qvel'] +
                 self.w_ee * rewards['end_effector'] +
                 self.w_com * rewards['com'] +
                 self.w_root * rewards['root_orient'])
        
        return total, rewards

reward_fn = MotionTrackingReward(model)
print("Reward function initialized.")
print(f"Weights: qpos={reward_fn.w_qpos}, qvel={reward_fn.w_qvel}, "
      f"ee={reward_fn.w_ee}, com={reward_fn.w_com}, root={reward_fn.w_root}")

In [ ]:
# Demonstrate the reward function
# Case 1: Perfect tracking (sim = ref) -> reward should be ~1.0
ref_data = mujoco.MjData(model)

# Set both to the same pose
test_pose = ref_qpos[50]  # pick a frame from reference
data.qpos[:] = test_pose
ref_data.qpos[:] = test_pose
data.qvel[:] = 0
ref_data.qvel[:] = 0

mujoco.mj_forward(model, data)
mujoco.mj_forward(model, ref_data)

total, components = reward_fn.compute(data, test_pose, np.zeros(model.nv), ref_data)
print("Case 1: Perfect tracking")
print(f"  Total reward: {total:.4f}")
for k, v in components.items():
    print(f"  {k:15s}: {v:.4f}")

print()

# Case 2: Moderate error
noisy_pose = test_pose.copy()
noisy_pose[7:] += np.random.randn(model.nq - 7) * 0.2  # add noise to joint angles
data.qpos[:] = noisy_pose
mujoco.mj_forward(model, data)

total2, components2 = reward_fn.compute(data, test_pose, np.zeros(model.nv), ref_data)
print("Case 2: Noisy joints (std=0.2 rad)")
print(f"  Total reward: {total2:.4f}")
for k, v in components2.items():
    print(f"  {k:15s}: {v:.4f}")

In [ ]:
# Visualize how reward decreases as error increases
errors = np.linspace(0, 2, 100)

fig, ax = plt.subplots(figsize=(8, 5))
for k, label in [(0.5, 'k=0.5 (loose)'), (2.0, 'k=2.0 (default)'), 
                  (5.0, 'k=5.0 (tight)'), (10.0, 'k=10.0 (very tight)')]:
    ax.plot(errors, np.exp(-k * errors**2), label=label)

ax.set_xlabel('Error (radians or meters)')
ax.set_ylabel('Reward')
ax.set_title('exp(-k * error²): Tracking Reward vs Error')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('imgs/reward_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Complete Motion Tracking Demo

Now let's put everything together in a complete motion tracking loop:

1. Load reference motion
2. At each timestep:
   - Look up the current reference pose
   - Use PD control to track the reference joint angles (simulating what an RL policy would output)
   - Step the simulation
   - Compute tracking reward
3. Render and log results

This demonstrates the **inner loop** that would be wrapped by an RL training framework.

In [ ]:
def run_motion_tracking(model, ref_times, ref_qpos, kp=100.0, kd=10.0, render_every=10):
    """Run a complete motion tracking simulation.
    
    Simulates a PD controller tracking the reference motion,
    computing rewards at each step.
    """
    data = mujoco.MjData(model)
    ref_data = mujoco.MjData(model)
    renderer_local = mujoco.Renderer(model, height=480, width=640)
    
    sim_dt = model.opt.timestep
    ref_dt = ref_times[1] - ref_times[0]
    steps_per_ref = max(1, int(ref_dt / sim_dt))
    
    reward_fn_local = MotionTrackingReward(model)
    
    # Initialize to first reference frame
    data.qpos[:] = ref_qpos[0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)
    
    kp_arr = np.full(model.nu, kp)
    kd_arr = np.full(model.nu, kd)
    gears = model.actuator_gear[:, 0]
    
    frames = []
    rewards_log = []
    component_log = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}
    
    total_steps = 0
    for ref_idx in range(len(ref_qpos) - 1):
        # Compute reference velocity from finite differences
        ref_qvel = np.zeros(model.nv)
        ref_qvel[6:] = (ref_qpos[ref_idx + 1, 7:] - ref_qpos[ref_idx, 7:]) / ref_dt
        
        # Set reference data for reward computation
        ref_data.qpos[:] = ref_qpos[ref_idx]
        mujoco.mj_forward(model, ref_data)
        
        # Target for PD controller = reference joint angles
        target_joints = ref_qpos[ref_idx, 7:]
        
        for sub_step in range(steps_per_ref):
            # PD control
            torques = pd_controller(model, data, target_joints, kp_arr, kd_arr)
            data.ctrl[:] = np.clip(torques / gears, -1, 1)
            
            mujoco.mj_step(model, data)
            total_steps += 1
            
            # Render
            if total_steps % render_every == 0:
                renderer_local.update_scene(data)
                frames.append(renderer_local.render().copy())
        
        # Compute reward at ref frame rate
        total_r, comp = reward_fn_local.compute(data, ref_qpos[ref_idx], ref_qvel, ref_data)
        rewards_log.append(total_r)
        for k in component_log:
            component_log[k].append(comp[k])
    
    renderer_local.close()
    return frames, np.array(rewards_log), {k: np.array(v) for k, v in component_log.items()}

print("Running motion tracking simulation...")
frames_track, rewards, components = run_motion_tracking(
    model, ref_times, ref_qpos, kp=300.0, kd=30.0, render_every=20
)
print(f"Done! {len(frames_track)} frames, avg reward: {rewards.mean():.4f}")

In [ ]:
media.show_video(frames_track, fps=25, title='PD tracking of reference motion')

In [ ]:
# Plot tracking rewards over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Total reward
axes[0].plot(ref_times[:len(rewards)], rewards)
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Motion Tracking Reward Over Time')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3)

# Component rewards
for name, values in components.items():
    axes[1].plot(ref_times[:len(values)], values, label=name)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Reward Component')
axes[1].set_title('Individual Reward Components')
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('imgs/tracking_rewards.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side comparison: reference (kinematic) vs tracked (physics)
# Re-run to capture both
data_sim = mujoco.MjData(model)
data_ref = mujoco.MjData(model)
renderer_sim = mujoco.Renderer(model, height=360, width=480)
renderer_ref = mujoco.Renderer(model, height=360, width=480)

data_sim.qpos[:] = ref_qpos[0]
data_sim.qvel[:] = 0
mujoco.mj_forward(model, data_sim)

kp_arr = np.full(model.nu, 300.0)
kd_arr = np.full(model.nu, 30.0)
gears = model.actuator_gear[:, 0]

ref_dt = ref_times[1] - ref_times[0]
sim_dt = model.opt.timestep
steps_per_ref = max(1, int(ref_dt / sim_dt))

comparison_frames = []
for ref_idx in range(0, len(ref_qpos) - 1, 3):
    target = ref_qpos[ref_idx, 7:]
    for _ in range(steps_per_ref * 3):
        torques = pd_controller(model, data_sim, target, kp_arr, kd_arr)
        data_sim.ctrl[:] = np.clip(torques / gears, -1, 1)
        mujoco.mj_step(model, data_sim)
    
    # Render simulated
    renderer_sim.update_scene(data_sim)
    img_sim = renderer_sim.render().copy()
    
    # Render reference
    data_ref.qpos[:] = ref_qpos[ref_idx]
    mujoco.mj_forward(model, data_ref)
    renderer_ref.update_scene(data_ref)
    img_ref = renderer_ref.render().copy()
    
    combined = np.concatenate([img_ref, img_sim], axis=1)
    comparison_frames.append(combined)

renderer_sim.close()
renderer_ref.close()

media.show_video(comparison_frames, fps=15, title='Left: Reference (kinematic) | Right: PD Tracked (physics)')

## 9. RL Integration: Observation, Action, and Episode Design

When wrapping the above into an RL environment (e.g., Gymnasium), the key design decisions are:

### Observation Space

The observation typically includes:

| Component | Description | Dimension |
|-----------|-------------|----------|
| Root height | `data.qpos[2]` | 1 |
| Root orientation | `data.qpos[3:7]` (quaternion) | 4 |
| Joint angles | `data.qpos[7:]` | nq - 7 |
| Root velocity | `data.qvel[0:6]` (linear + angular) | 6 |
| Joint velocities | `data.qvel[6:]` | nv - 6 |
| **Reference target** | relative reference pose at current phase | varies |
| Phase variable | where we are in the reference motion cycle | 1-2 |

**Important**: the reference target is often expressed as the **difference** between the current pose and the reference, or as the reference itself. Including future reference frames (look-ahead) can also help.

### Action Space

| Type | Description | Dimension |
|------|-------------|----------|
| PD targets | Target joint angles → PD controller computes torques | nu |
| Residual PD | action = offset added to reference angles | nu |
| Direct torque | Normalized torques applied directly | nu |

**Residual PD** is popular: `target = ref_qpos[7:] + action * scale`. The policy learns corrections to the reference.

### Episode Design

- **Reference State Initialization (RSI)**: randomly sample a starting frame from the reference motion and initialize qpos/qvel accordingly. This is crucial for learning — without RSI, the policy only learns to track from the beginning.
- **Early termination**: end the episode if the humanoid falls (e.g., torso height drops below threshold, or contacts indicate a fall).
- **Cyclic motions**: for locomotion, the reference motion loops. The phase variable tracks position within the cycle.

In [ ]:
class MotionTrackingEnv:
    """Minimal motion tracking RL environment skeleton.
    
    Demonstrates how to structure the observation/action/reward
    for motion imitation RL. Not a full Gymnasium env, but shows the key pieces.
    """
    
    def __init__(self, model_xml, ref_qpos, ref_times,
                 kp=300.0, kd=30.0, action_scale=0.3, ctrl_dt=0.02):
        self.model = mujoco.MjModel.from_xml_string(model_xml)
        self.data = mujoco.MjData(self.model)
        self.ref_data = mujoco.MjData(self.model)
        
        self.ref_qpos = ref_qpos
        self.ref_times = ref_times
        self.ref_dt = ref_times[1] - ref_times[0]
        self.duration = ref_times[-1]
        self.n_ref_frames = len(ref_times)
        
        self.sim_dt = self.model.opt.timestep
        self.ctrl_dt = ctrl_dt
        self.sim_steps_per_ctrl = max(1, int(ctrl_dt / self.sim_dt))
        
        self.kp = np.full(self.model.nu, kp)
        self.kd = np.full(self.model.nu, kd)
        self.gears = self.model.actuator_gear[:, 0]
        self.action_scale = action_scale
        
        self.reward_fn = MotionTrackingReward(self.model)
        
        # Dimensions
        self.obs_dim = self._get_obs().shape[0] if False else None  # computed in reset
        self.act_dim = self.model.nu
        
        self.phase = 0.0
        self.time = 0.0
    
    def reset(self, start_frame=None):
        """Reset environment with Reference State Initialization."""
        if start_frame is None:
            start_frame = np.random.randint(0, self.n_ref_frames)
        
        self.phase = start_frame / self.n_ref_frames
        self.time = self.ref_times[start_frame]
        self.current_ref_idx = start_frame
        
        # Initialize to reference pose (with small noise for robustness)
        self.data.qpos[:] = self.ref_qpos[start_frame]
        self.data.qvel[:] = 0
        mujoco.mj_forward(self.model, self.data)
        
        return self._get_obs()
    
    def _get_obs(self):
        """Construct the observation vector."""
        ref_idx = self.current_ref_idx % self.n_ref_frames
        ref_pose = self.ref_qpos[ref_idx]
        
        obs = np.concatenate([
            # Proprioception
            [self.data.qpos[2]],          # root height
            self.data.qpos[3:7],           # root orientation (quaternion)
            self.data.qpos[7:],            # joint angles
            self.data.qvel[:6],            # root velocity
            self.data.qvel[6:],            # joint velocities
            # Reference target
            ref_pose[7:] - self.data.qpos[7:],  # joint angle error to reference
            ref_pose[3:7],                       # reference root orientation
            # Phase
            [np.sin(2 * np.pi * self.phase),
             np.cos(2 * np.pi * self.phase)],    # cyclic phase encoding
        ])
        return obs
    
    def step(self, action):
        """Take one environment step.
        
        Action: residual joint angle offsets added to the reference.
        """
        ref_idx = self.current_ref_idx % self.n_ref_frames
        ref_pose = self.ref_qpos[ref_idx]
        
        # Residual PD: target = reference + action * scale
        target_joints = ref_pose[7:] + action * self.action_scale
        
        # Run simulation
        for _ in range(self.sim_steps_per_ctrl):
            torques = self.kp * (target_joints - self.data.qpos[7:]) - self.kd * self.data.qvel[6:]
            self.data.ctrl[:] = np.clip(torques / self.gears, -1, 1)
            mujoco.mj_step(self.model, self.data)
        
        # Advance phase
        self.time += self.ctrl_dt
        self.current_ref_idx += max(1, int(self.ctrl_dt / self.ref_dt))
        self.phase = (self.current_ref_idx % self.n_ref_frames) / self.n_ref_frames
        
        # Compute reward
        ref_idx_new = self.current_ref_idx % self.n_ref_frames
        ref_qvel = np.zeros(self.model.nv)
        if ref_idx_new + 1 < self.n_ref_frames:
            ref_qvel[6:] = (self.ref_qpos[ref_idx_new + 1, 7:] - self.ref_qpos[ref_idx_new, 7:]) / self.ref_dt
        
        self.ref_data.qpos[:] = self.ref_qpos[ref_idx_new]
        mujoco.mj_forward(self.model, self.ref_data)
        
        reward, reward_info = self.reward_fn.compute(
            self.data, self.ref_qpos[ref_idx_new], ref_qvel, self.ref_data
        )
        
        # Early termination: check if fallen
        torso_height = self.data.qpos[2]
        done = torso_height < 0.6  # fallen if torso too low
        
        obs = self._get_obs()
        return obs, reward, done, reward_info

# Create and test the environment
env = MotionTrackingEnv(HUMANOID_XML, ref_qpos, ref_times)
obs = env.reset(start_frame=0)
print(f"Observation dimension: {obs.shape[0]}")
print(f"Action dimension: {env.act_dim}")
print(f"Observation breakdown:")
print(f"  Root height:     1")
print(f"  Root quat:       4")
print(f"  Joint angles:    {model.nq - 7}")
print(f"  Root velocity:   6")
print(f"  Joint velocity:  {model.nv - 6}")
print(f"  Joint ref error: {model.nq - 7}")
print(f"  Ref root quat:   4")
print(f"  Phase (sin,cos): 2")
print(f"  Total:           {1+4+(model.nq-7)+6+(model.nv-6)+(model.nq-7)+4+2}")

In [ ]:
# Run a short episode with random actions to show the environment works
obs = env.reset(start_frame=0)
episode_rewards = []
episode_info = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}

renderer_env = mujoco.Renderer(env.model, height=480, width=640)
env_frames = []

for step in range(100):
    # Random action (in practice, this would come from the RL policy)
    action = np.random.randn(env.act_dim) * 0.1  # small random perturbations
    
    obs, reward, done, info = env.step(action)
    episode_rewards.append(reward)
    for k in episode_info:
        episode_info[k].append(info[k])
    
    # Render
    renderer_env.update_scene(env.data)
    env_frames.append(renderer_env.render().copy())
    
    if done:
        print(f"Episode terminated at step {step} (humanoid fell)")
        break

renderer_env.close()

print(f"Episode length: {len(episode_rewards)} steps")
print(f"Mean reward: {np.mean(episode_rewards):.4f}")
print(f"Min/Max reward: {np.min(episode_rewards):.4f} / {np.max(episode_rewards):.4f}")

media.show_video(env_frames, fps=25, title='RL env episode with random actions')

In [ ]:
# Run a "zero action" episode — pure reference tracking via PD
obs = env.reset(start_frame=0)
zero_rewards = []
zero_info = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}
zero_frames = []
renderer_z = mujoco.Renderer(env.model, height=480, width=640)

for step in range(100):
    action = np.zeros(env.act_dim)  # zero residual -> pure reference tracking
    obs, reward, done, info = env.step(action)
    zero_rewards.append(reward)
    for k in zero_info:
        zero_info[k].append(info[k])
    
    renderer_z.update_scene(env.data)
    zero_frames.append(renderer_z.render().copy())
    
    if done:
        print(f"Episode terminated at step {step}")
        break

renderer_z.close()

print(f"Zero-action episode: {len(zero_rewards)} steps, mean reward: {np.mean(zero_rewards):.4f}")

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_rewards, label='Random actions', alpha=0.8)
ax.plot(zero_rewards, label='Zero actions (pure PD tracking)', alpha=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Reward')
ax.set_title('Episode Reward: Random vs Zero Actions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('imgs/episode_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

media.show_video(zero_frames, fps=25, title='Zero-action episode (pure PD reference tracking)')

## Summary

This tutorial covered the MuJoCo fundamentals needed for motion tracking RL:

| Concept | Key MuJoCo API | Section |
|---------|---------------|--------|
| Model structure | `model.njnt`, `model.nbody`, `model.nu` | §1 |
| State access | `data.qpos`, `data.qvel` | §2 |
| Forward kinematics | `data.xpos`, `data.xquat`, `data.subtree_com` | §3 |
| Quaternion math | `mju_mulQuat`, `mju_negQuat`, `mju_quat2Vel`, `mju_subQuat` | §4 |
| Actuator control | `data.ctrl`, `model.actuator_gear` | §5 |
| Reference motion | Direct `qpos` setting + `mj_forward` | §6 |
| Tracking reward | `exp(-k * error²)` with multiple components | §7 |
| Full tracking loop | PD control + reward computation + rendering | §8 |
| RL env design | Observation/action/episode structure | §9 |

### Next steps toward a full BeyondMimic-style system

1. **RL training**: Use PPO/SAC from libraries like [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) or [RSL-RL](https://github.com/leggedrobotics/rsl_rl)
2. **Real MoCap data**: Load `.bvh`/`.c3d` files and retarget to the MuJoCo humanoid
3. **Domain randomization**: Randomize dynamics parameters for sim-to-real transfer
4. **Adversarial training (AMP)**: Replace hand-crafted rewards with a learned discriminator
5. **Diffusion policy distillation**: Train a diffusion model on successful tracking rollouts (BeyondMimic approach)